In [ ]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [1]:
# Clear filesystem cache (pagecache, dentries, inodes)
!sudo sync
!sudo sh -c 'echo 3 > /proc/sys/vm/drop_caches'

sh: 1: cannot create /proc/sys/vm/drop_caches: Read-only file system


In [2]:
# Check GPU status
!nvidia-smi

Thu Apr  2 14:33:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:83:00.0 Off |                  N/A |
|  0%   52C    P8             44W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Data configuration
DATA_ID = 'alxxtexxr/BVIR-Data'

# k-means configuration
SEED = 42

# For small experiment
# DATA_ALLOW_DIR = 'model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_100.data_rl_alxxtexxr_ViRL1.25K_100'
# K = 1024

# For larger experiment
# DATA_ALLOW_DIR = 'model_unsloth_Qwen3-VL-8B-Instruct_fp16.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000'
DATA_ALLOW_DIR = 'model_unsloth_Qwen3-VL-8B-Thinking_fp16.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000'
K = 4096 # 4096 | 8192

In [4]:
from pathlib import Path
from huggingface_hub import snapshot_download

data_dir = snapshot_download(
    repo_id=DATA_ID,
    repo_type='dataset',
    allow_patterns=f'{DATA_ALLOW_DIR}/**',
    local_dir='data',
    local_dir_use_symlinks=False,
)
vision_data_dir = Path(data_dir) / DATA_ALLOW_DIR

print("Vision data downloaded to:", vision_data_dir)

/venv/main/lib/python3.12/site-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

model_unsloth_Qwen3-VL-8B-Thinking_fp16.(…):   0%|          | 0.00/134M [00:00<?, ?B/s]

model_unsloth_Qwen3-VL-8B-Thinking_fp16.(…):   0%|          | 0.00/134M [00:00<?, ?B/s]

Vision data downloaded to: /workspace/BVIR/data/model_unsloth_Qwen3-VL-8B-Thinking_fp16.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000


In [5]:
# %%capture
%pip install faiss-gpu-cu12

Note: you may need to restart the kernel to use updated packages.


In [6]:
from pathlib import Path
import numpy as np
import faiss

# Configuration parameters
DATA_DIR = Path('data')
# DIM = 3584
DIM = 4096
NITER = 20

# Set random seeds for reproducibility
np.random.seed(SEED)
# faiss.rand.seed(SEED) # AttributeError: 'function' object has no attribute 'seed'

# Load all batch file paths
batch_paths = [f for f in vision_data_dir.glob('*.npz')]

# Initialize k-means (with GPU support)
kmeans = faiss.Kmeans(
    d=DIM,
    k=K,
    niter=NITER,
    verbose=True,
    gpu=True,
    seed=SEED,
    min_points_per_centroid=10,
)

In [7]:
# Collect training samples
init_buf = []

print("Collecting training samples...")

for batch_path in batch_paths:
    batch = np.load(batch_path, mmap_mode='r')
    visual_embs = batch['visual_embs'].astype(np.float32) # (32400, 2048)

    faiss.normalize_L2(visual_embs)
    init_buf.append(visual_embs)

train_data = np.vstack(init_buf)

print("Training data shape:", train_data.shape)
assert train_data.shape[0] >= K and train_data.shape[1] == DIM, "Training data must have at least K samples and match the specified dimension."

Training data shape: (512000, 4096)


In [8]:
# Train (ONCE)
print("Training k-means...")
kmeans.train(train_data)

Training k-means...
Clustering 512000 points in 4096D to 4096 clusters, redo 1 times, 20 iterations
  Preprocessing in 2.25 s
  Iteration 19 (82.23 s, search 63.89 s): objective=201033 imbalance=1.172 nsplit=0       


np.float64(201032.703125)

In [9]:
# Save codebook
codebook = kmeans.centroids
print("Visual codebook shape:", codebook.shape)

npy_path = f'codebook_faiss_{K}.npy'
np.save(npy_path, codebook)
print("Visual codebook NumPy array saved to:", npy_path)

Visual codebook shape: (4096, 4096)
Visual codebook NumPy array saved to: codebook_faiss_4096.npy


In [10]:
# Build search index (Optional)
# For later quantization
index = faiss.IndexFlatIP(DIM) # Inner product = cosine
index.add(codebook)

index_path = f'codebook_faiss_{K}.index'
faiss.write_index(index, index_path)
print("Visual codebook FAISS index saved to:", index_path)

Visual codebook FAISS index saved to: codebook_faiss_4096.index


In [11]:
print("Min centroid norm:", np.linalg.norm(codebook, axis=1).min())

Min centroid norm: 0.4550378


In [12]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj=npy_path,
    path_in_repo=f'{DATA_ALLOW_DIR}/{npy_path}',
    repo_id=DATA_ID,
    repo_type='dataset',
)
upload_file(
    path_or_fileobj=index_path,
    path_in_repo=f'{DATA_ALLOW_DIR}/{index_path}',
    repo_id=DATA_ID,
    repo_type='dataset',
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/BVIR-Data/commit/40bb895b943071ab58e26cdb52ffba0d97f09a16', commit_message='Upload model_unsloth_Qwen3-VL-8B-Thinking_fp16.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_4096.index with huggingface_hub', commit_description='', oid='40bb895b943071ab58e26cdb52ffba0d97f09a16', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/BVIR-Data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/BVIR-Data'), pr_revision=None, pr_num=None)